In [9]:
import os
from pathlib import Path

import plotly.express as px
import torch
import torch.nn.functional as F
from dotenv import load_dotenv

from analysis.utils import load_autoencoder
from koopmann import aesthetics
from koopmann.llm import (
    HF_LLM_DICT,
    LMHiddenStatesDataset,
    extract_hidden_states_from_hf,
    get_hf_llm,
    read_prompts,
)
from koopmann.llm.model import inject_token_state
from koopmann.utils import get_device
from koopmann.visualization import plot_eigenvalues

assert load_dotenv(Path.cwd().parent / ".env")

WEIGHTS_CACHE = os.getenv("WEIGHTS_CACHE")
assert WEIGHTS_CACHE is not None

HF_HOME = os.getenv("HF_HOME")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
device = get_device()
model = HF_LLM_DICT["llama_3p2_1b"]
k = 1
dim = 512
seed = 22
layer_i = 10
layer_j = 15

autoencoder, metadata = load_autoencoder(
    WEIGHTS_CACHE,
    f"dim_{dim}_k_{k}_autoencoder_dummy_seed_{seed}",
)
_ = autoencoder.to(device).eval()

hf_model, hf_tokenizer = get_hf_llm(
    hf_name=model,
    cache_dir=HF_HOME,
    device=device,
)

_ = hf_model.to(device).eval()

/home/dunstan/git/koopmann/koopmann/models/autoencoder.py:52: UserWarning: The latent dimension 512 should probably be larger than the input dimension 2048!
  warnings.warn(


In [5]:
# -----------------------------
# 1. Extract hidden states
# -----------------------------
prompt = "I come from"
hs, mask = extract_hidden_states_from_hf(
    model=hf_model, tokenizer=hf_tokenizer, prompts=[prompt], device=device
)

# hs: (1, P, L+1, D), mask: (1, P)
last_tok = int(mask[0].sum().item() - 1)  # last non-pad token index

x_i = hs[0, last_tok, layer_i].float().to(device).unsqueeze(0)  # (1, D)
x_j = hs[0, last_tok, layer_j].float().to(device).unsqueeze(0)  # (1, D)

In [6]:
# -----------------------------
# 2. KAE prediction
# -----------------------------
with torch.no_grad():
    x_j_hat, _ = autoencoder.forward(x_i)  # (1, D)
    x_j_hat = x_j_hat.squeeze(0)

In [7]:
# -----------------------------
# 3. Offline fidelity metrics
# -----------------------------
cos = F.cosine_similarity(x_j, x_j_hat).item()

print(f"Prompt: '{prompt}'")
print(f"Cosine Sim:  {cos:.6f}")

Prompt: 'I come from'
Cosine Sim:  0.126685


In [10]:
max_new_tokens = 10

# tokenize prompt
tok = hf_tokenizer(prompt, return_tensors="pt")
tok = {k: v.to(device) for k, v in tok.items()}

input_ids_orig = tok["input_ids"]
attn_mask_orig = tok["attention_mask"]

# make sure injection vector is on the right device
replacement_vec = x_j_hat
if isinstance(replacement_vec, torch.Tensor):
    replacement_vec = replacement_vec.to(device)


# -----------------------------
# Greedy generation with optional injection
# -----------------------------
def generate_with_optional_injection(tok, inject: bool, max_new_tokens: int):
    """
    Greedy generation with optional KAE injection at layer_j
    on the last token at each decoding step.
    """
    input_ids = tok["input_ids"].clone()
    attention_mask = tok["attention_mask"].clone()

    for _ in range(max_new_tokens):
        last_tok_idx = input_ids.size(1) - 1  # position to overwrite

        model_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )

        if inject:
            # inject into the last token at layer_j for this forward pass
            with inject_token_state(
                hf_model,
                layer_j=layer_j,
                token_idx=last_tok_idx,
                new_vec=replacement_vec,
            ):
                out = hf_model(**model_kwargs)
        else:
            out = hf_model(**model_kwargs)

        # greedy next-token selection
        next_token_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)  # [B, 1]

        # append next token
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)

        # extend attention mask
        new_mask = attention_mask.new_ones((attention_mask.size(0), 1))
        attention_mask = torch.cat([attention_mask, new_mask], dim=-1)

    return input_ids


# -----------------------------
# 1) Baseline generation
# -----------------------------
gen_ids_base = generate_with_optional_injection(
    tok, inject=False, max_new_tokens=max_new_tokens
)

# -----------------------------
# 2) KAE-injected generation
# -----------------------------
gen_ids_kae = generate_with_optional_injection(
    tok, inject=True, max_new_tokens=max_new_tokens
)

# -----------------------------
# 3) Decode and compare
# -----------------------------
prompt_len = input_ids_orig.size(1)

baseline_continuation = hf_tokenizer.decode(
    gen_ids_base[0, prompt_len:], skip_special_tokens=True
)
kae_continuation = hf_tokenizer.decode(
    gen_ids_kae[0, prompt_len:], skip_special_tokens=True
)

print("\nOnline Multi-Token Fidelity Test")
print("--------------------------------")
print(f"Prompt: {prompt!r}\n")
print(f"Baseline continuation ({max_new_tokens} tokens):")
print(baseline_continuation)
print("\nKAE continuation ({max_new_tokens} tokens):")
print(kae_continuation)


Online Multi-Token Fidelity Test
--------------------------------
Prompt: 'I come from'

Baseline continuation (10 tokens):
 a family of farmers. My grandfather was a farmer

KAE continuation ({max_new_tokens} tokens):
 Japaneseouns Japanese Japanese Japanese Japanese Japanese Japanese Japanese Japanese
